### University of Virginia
### DS 5110: Big Data Systems

### Lab: Supervised Learning
### Last updated: March 2, 2023

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [15]:
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

#### Enter code and solution

In [16]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pyspark.sql.functions as F
import pyspark.sql.types as typ

# Read data in from csv
all_data = spark.read.csv(DATA_FILEPATH, inferSchema=True, header=True)

In [17]:
# Turn nominal column into binary integer column for logistic regression

recode_dictionary = {
    'diagnosis': {
        'M': 1,
        'B': 0
    }
}

def recode(col, key):
    return recode_dictionary[key][col]
rec_integer = F.udf(recode, typ.IntegerType())

data = all_data.withColumn(
    'diagnosis_code', 
        rec_integer(
            'diagnosis', F.lit('diagnosis')
        ))

data = data.select(['id', 'diagnosis', 'diagnosis_code', 'f1', 'f2'])

In [18]:
# Split data into training (60%) and test (40%) set
rsplits = data.randomSplit([0.6, 0.4], 314)
train = rsplits[0]
test = rsplits[1]

In [19]:
# Convert data to vector format, grabbing f1 and f2 cols as predictors
assembler = VectorAssembler(inputCols=['f1', 'f2'], outputCol='features')
tr_assembled = assembler.transform(train)
te_assembled = assembler.transform(test)

In [20]:
# Use scalar to standardize predictors
scaler = StandardScaler(inputCol='features', outputCol='sFeatures')
scalerModel = scaler.fit(tr_assembled)

tr_scaled = scalerModel.transform(tr_assembled)
te_scaled = scalerModel.transform(te_assembled)

In [21]:
# Build logistic regression model 
lr_model = LogisticRegression(
    labelCol = 'diagnosis_code',
    featuresCol = 'sFeatures',
    maxIter = 10)

lr_fit = lr_model.fit(tr_scaled)

# Print the coefficients and intercept for logistic regression
print("Coefficients: " + str(lr_fit.coefficients))
print("Intercept: " + str(lr_fit.intercept))

Coefficients: [3.5700446280343394,1.0818379381825265]
Intercept: -19.50325708863459


In [22]:
# Predict on test data

lr_predictions = lr_fit.transform(te_scaled)

In [33]:
# Evaluate model using ROC 
# source: https://matplotlib.org/stable/gallery/text_labels_and_annotations/placing_text_boxes.html
# source: https://medium.com/@demrahayan/evaluating-binary-classification-models-with-pyspark-2afc5ac7937f

import matplotlib.pyplot as plt

evaluator = BinaryClassificationEvaluator(rawPredictionCol='probability', labelCol='diagnosis_code')
auc = evaluator.evaluate(lr_predictions)
print(f'Test set AUC: {auc}')

Test set AUC: 0.9514088556641738


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [11]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [12]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution

In [13]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Read data in from csv
reg_all_data = spark.read.csv(DATA_FILEPATH2, inferSchema=True, header=True)

In [25]:
data_lin = reg_all_data \
    .withColumn('median_house_value_scaled', reg_all_data['median_house_value']/100000) \
    .withColumn('rooms_per_household', reg_all_data['total_rooms']/reg_all_data['households'])

splits_lin = data_lin.randomSplit([0.8,0.2], 314)
train_lin = splits_lin[0]
test_lin = splits_lin[1]

feats = ["total_bedrooms", "population", "households", "median_income", "rooms_per_household"]

In [26]:
# Transform predictors into vector form
assembler_lin = VectorAssembler(inputCols=feats, outputCol="features")

tr_assembled_lin = assembler_lin.transform(train_lin)
te_assembled_lin = assembler_lin.transform(test_lin)

In [27]:
# Scale predictors
scaler_lin = StandardScaler(inputCol="features", outputCol="sFeatures")

tr_scalerModel_lin = scaler_lin.fit(tr_assembled_lin)

tr_scaled_lin= tr_scalerModel_lin.transform(tr_assembled_lin)
te_scaled_lin = tr_scalerModel_lin.transform(te_assembled_lin)

In [28]:
linr_model = LinearRegression(
    labelCol = 'median_house_value_scaled',
    featuresCol = 'sFeatures',
    maxIter = 10,
    regParam = 0.3,
    elasticNetParam=0.8) 

linr_fit = linr_model.fit(tr_scaled_lin)

# Print the weights and intercept for linear regression
print("Weights: " + str(linr_fit.coefficients))
print("Intercept: " + str(linr_fit.intercept))

Weights: [0.0,0.0,0.0,0.5228383534770863,0.0,0.0]
Intercept: 0.9991283260557133


In [29]:
linr_pred = linr_fit.transform(te_scaled_lin)
evaluator_lin = RegressionEvaluator(predictionCol="prediction", labelCol="median_house_value_scaled")
mse =  evaluator_lin.evaluate(linr_pred, {evaluator_lin.metricName: "mse"})
rsq =  evaluator_lin.evaluate(linr_pred, {evaluator_lin.metricName: "r2"})

print("Mean Squared Error:", mse)
print("R Squared", rsq)

Mean Squared Error: 0.7551748008242244
R Squared 0.4347278249259109
